# Fase 1 - Baseline e Modelagem Classica

In [1]:
import polars as pl
import numpy as np 
from sklearn.model_selection import train_test_split

gold = pl.read_parquet("../data/processed/gold/gold_account_activity.parquet")
print(f"Shape: {gold.shape}")
print(f"Taxa de churne: {gold['is_churn'].mean():.1%}")

Shape: (25000, 21)
Taxa de churne: 9.0%


In [2]:
df  = gold.to_pandas()

train_val, test = train_test_split(
    df, test_size=0.15, stratify=df["is_churn"], random_state=42
)
train, val = train_test_split(
    train_val, test_size=0.15 / 0.85, stratify=train_val["is_churn"], random_state=42
)

print(f"Treino:     {len(train):>6} contas ({train['is_churn'].mean():.1%} churn)")
print(f"Validação:  {len(val):>6} contas ({val['is_churn'].mean():.1%} churn)")
print(f"Teste:      {len(test):>6} contas ({test['is_churn'].mean():.1%} churn)")

Treino:      17499 contas (9.0% churn)
Validação:    3751 contas (9.0% churn)
Teste:        3750 contas (9.0% churn)


In [3]:
target = "is_churn"

features_categoricas = ["faixa_etaria", "gender", "registered_via"]
features_booleanas = [
    "tem_cadastro", "tem_uso_registrado", "auto_renew_ultima",
    "desligou_auto_renovacao", "ja_cancelou"
]
features_numericas = [
    "tenure_cadastro_dias", "n_transacoes", "desconto_medio", "plano_dias_ultimo",
    "dias_desde_ultima_transacao", "total_secs_ultimo_mes", "variacao_uso_mes",
    "tendencia_uso_3m", "meses_ativos", "n_tickets_total", "n_tickets_ultimos_30d",
]
todas_features = features_categoricas + features_booleanas + features_numericas

X_train, y_train = train[todas_features], train[target]
X_val, y_val = val[todas_features], val[target]
X_test, y_test = test[todas_features], test[target]

print(f"Total de features: {len(todas_features)}")
print(f"  Categóricas: {len(features_categoricas)} | Booleanas: {len(features_booleanas)} | Numéricas: {len(features_numericas)}")

Total de features: 19
  Categóricas: 3 | Booleanas: 5 | Numéricas: 11


In [ ]:
import numpy as np

print("Colunas com infinito:")
print(X_train[features_numericas].isin([np.inf, -np.inf]).sum())

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

preprocessador = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas),
    ("bool", SimpleImputer(strategy="most_frequent"), features_booleanas),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), features_numericas),
])

modelo_lr = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", LogisticRegression(max_iter=1000, random_state=42))
])

modelo_lr.fit(X_train, y_train)
print("Logistic Regression treinado")

ValueError: Input X contains infinity or a value too large for dtype('float64').